# Instalação de bibliotecas

In [0]:
!pip install feature_engine

# Importações de bibliotecas

In [0]:
import mlflow

# Importação do modelo

In [0]:
# IMPORTANDO MODELO
model = mlflow.sklearn.load_model("models:/workspace.tmw_trabalhofinal.model_trabalhofinal/2")



# Importação dos dados

In [0]:
# IMPORTANDO DADOS
df = spark.sql(""" 
    SELECT  * EXCEPT (t2.dtRef, t2.idCliente)
    FROM    workspace.tmw_trabalhofinal.fs_pontos AS t1
            LEFT JOIN workspace.tmw_trabalhofinal.fs_cursos AS t2
                ON t1.IdCliente = t2.idCliente
                AND t1.dtRef = t2.dtRef
    WHERE   t1.dtRef = (
                SELECT  MAX(dtRef)
                FROM    workspace.tmw_trabalhofinal.fs_pontos
            )
""").toPandas()

df

# Predição de churn para o mês de 01/07/2026

In [0]:
X = df[model.feature_names_in_]
predict = model.predict_proba(X)[:, 0]

df['prob_ativacao'] = predict
df['prob_churn'] = 1 - predict
df[['IdCliente', 'prob_churn']].sort_values('prob_churn', ascending=False).reset_index(drop=True).head(50)